# SWOT PIXC tile visualizer, Earthdata search, and download

This training notebook demonstrates a simple workflow for identifying and downloading **SWOT PIXC** products.

The notebook is organized into three main steps:

1. Display the SWOT High-Rate tile grid on an interactive map and identify the tile(s) covering a selected location.
2. Build a PIXC granule-name pattern and search NASA Earthdata.
3. Download the selected PIXC granules to a platform-independent course directory.

> **Training note:** The PIXC filename contains information such as cycle, pass, and tile reference. The interactive map helps identify the `TILE_REF` and `PASS_NUM` values needed to construct a granule search pattern.


## 1. Identify SWOT tiles with an interactive map

Place a marker on the map at the location of interest.

The notebook checks which SWOT High-Rate tile polygons intersect the selected point and displays:

- `TILE_REF`: the SWOT tile reference, such as `165R`;
- `PASS_NUM`: the corresponding SWOT pass number.

The tile grid is read from `swot_hr_mask.shp`, stored in the general course directory.


In [2]:
from pathlib import Path
import json

import geopandas as gpd
from ipyleaflet import Map, DrawControl, GeoJSON, Popup
from IPython.display import display
from ipywidgets import HTML
from shapely.geometry import Point, shape


# ============================================================
# USER CONFIGURATION
# ============================================================

# General directory used by the SWOT training course.
COURSE_DIR = Path.home() / "course" / "data"

# SWOT High-Rate tile grid.
SWOT_TILE_GRID = (
    COURSE_DIR
    / "swot_hr_mask.shp"
)


# ============================================================
# LOAD THE SWOT TILE GRID
# ============================================================

gdf_tiles = gpd.read_file(
    SWOT_TILE_GRID
)


if gdf_tiles.empty:

    raise ValueError(
        "The SWOT tile-grid shapefile is empty."
    )


# Interactive web maps use longitude/latitude coordinates.
if gdf_tiles.crs is None:

    print(
        "[WARNING] SWOT tile-grid CRS is undefined. "
        "Assuming EPSG:4326."
    )

    gdf_tiles = gdf_tiles.set_crs(
        epsg=4326
    )

else:

    gdf_tiles = gdf_tiles.to_crs(
        epsg=4326
    )


# Check that the fields required by this exercise exist.
required_columns = {
    "TILE_REF",
    "PASS_NUM",
}


missing_columns = (
    required_columns
    - set(gdf_tiles.columns)
)


if missing_columns:

    raise ValueError(
        "The tile-grid file is missing required columns: "
        f"{sorted(missing_columns)}"
    )


print(
    f"SWOT tile polygons loaded: {len(gdf_tiles):,}"
)


# ============================================================
# CREATE THE INTERACTIVE MAP
# ============================================================

# Approximate map center over Vietnam.
m = Map(
    center=(16.0, 106.0),
    zoom=5,
    layout={
        "height": "600px",
        "width": "1000px",
    },
    scroll_wheel_zoom=True
)


# Store temporary polygon and popup layers added after a click.
temporary_layers = []


def clear_temporary_layers():
    """
    Remove tile outlines and labels created by the previous marker.
    """

    global temporary_layers


    for layer in temporary_layers:

        try:

            m.remove_layer(
                layer
            )

        except Exception:
            pass


    temporary_layers = []


def handle_draw(
    target,
    action,
    geo_json
):
    """
    Process marker events from the interactive map.

    When a marker is created, the function identifies every SWOT tile
    intersecting the selected location and displays the tile reference
    and pass number.
    """

    if action == "deleted":

        clear_temporary_layers()

        return


    if action != "created":

        return


    clear_temporary_layers()


    # --------------------------------------------------------
    # GET MARKER COORDINATES
    # --------------------------------------------------------

    geometry = geo_json[
        "geometry"
    ]


    marker_coords = geometry[
        "coordinates"
    ]


    lon = marker_coords[0]
    lat = marker_coords[1]


    print(
        f"Selected location: "
        f"longitude={lon:.5f}, latitude={lat:.5f}"
    )


    point = Point(
        lon,
        lat
    )


    # --------------------------------------------------------
    # FIND SWOT TILES AT THE SELECTED LOCATION
    # --------------------------------------------------------

    # `intersects()` is used instead of `contains()` so that a point
    # exactly on a tile boundary is not accidentally ignored.
    selected_tiles = gdf_tiles[
        gdf_tiles.intersects(
            point
        )
    ].copy()


    if selected_tiles.empty:

        print(
            "No SWOT High-Rate tile intersects this location."
        )

        return


    # Display a compact table in the notebook output.
    tile_table = (
        selected_tiles[
            [
                "TILE_REF",
                "PASS_NUM",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "PASS_NUM",
                "TILE_REF",
            ]
        )
    )


    print()

    print(
        "SWOT tiles found:"
    )

    print(
        tile_table.to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # DISPLAY SELECTED TILE POLYGONS
    # --------------------------------------------------------

    geojson_data = json.loads(
        selected_tiles.to_json()
    )


    for feature in geojson_data[
        "features"
    ]:


        properties = feature[
            "properties"
        ]


        tile_ref = properties.get(
            "TILE_REF",
            "N/A"
        )


        pass_num = properties.get(
            "PASS_NUM",
            "N/A"
        )


        # Draw the selected tile with a red outline.
        tile_layer = GeoJSON(
            data=feature,
            style={
                "color": "red",
                "fillColor": "transparent",
                "weight": 2,
                "fillOpacity": 0,
            }
        )


        m.add_layer(
            tile_layer
        )

        temporary_layers.append(
            tile_layer
        )


        # Use a representative point for the label.
        # Unlike a polygon centroid, this point is guaranteed
        # to lie inside the polygon.
        polygon_shape = shape(
            feature["geometry"]
        )


        label_point = (
            polygon_shape
            .representative_point()
        )


        label = HTML(
            f"""
            <div style="font-size: 13px;">
                <b>Tile:</b> {tile_ref}<br>
                <b>Pass:</b> {pass_num}
            </div>
            """
        )


        popup = Popup(
            location=(
                label_point.y,
                label_point.x
            ),
            child=label,
            close_button=False,
            auto_close=False,
            close_on_escape_key=False
        )


        m.add_layer(
            popup
        )

        temporary_layers.append(
            popup
        )


# ============================================================
# ADD THE MARKER TOOL
# ============================================================

# Only the marker tool is enabled for this exercise.
draw_control = DrawControl(
    marker={
        "shapeOptions": {
            "color": "#FF0000"
        }
    },
    polyline={},
    polygon={},
    circle={},
    rectangle={},
    circlemarker={}
)


draw_control.on_draw(
    handle_draw
)


m.add_control(
    draw_control
)


# Display the interactive map.
display(
    m
)


SWOT tile polygons loaded: 86,590


Map(center=[16.0, 106.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

## 2. Search NASA Earthdata for SWOT PIXC granules

After identifying the tile and pass, build a PIXC granule-name pattern.

A typical PIXC granule begins with information in the following order:

```text
SWOT_L2_HR_PIXC_<cycle>_<pass>_<tile>...
```

For example:

```text
SWOT_L2_HR_PIXC_009_014_165R*
```

means:

- cycle `009`;
- pass `014`;
- tile `165R`;
- `*` accepts the remaining filename characters.

Multiple patterns can be included in `GRANULE_PATTERNS` when more than one tile or pass is required.


In [4]:
import earthaccess


# ============================================================
# EARTHDATA LOGIN
# ============================================================

# Reuse Earthdata credentials in future notebook sessions.
earthaccess.login(
    persist=True
)


# ============================================================
# SEARCH CONFIGURATION
# ============================================================

# Add one or more PIXC filename patterns.
#
# Pattern structure:
#
# SWOT_L2_HR_PIXC_<cycle>_<pass>_<tile>*
#
# Example:
#
# cycle = 009
# pass  = 014
# tile  = 165R
GRANULE_PATTERNS = [
    "SWOT_L2_HR_PIXC_050_549_179R*",
]


# Optional temporal filter.
#
# Set TEMPORAL = None to search all available dates
# matching the granule-name pattern.
TEMPORAL = None

# Example:
# TEMPORAL = ("2025-01-01", "2026-08-17")


# ============================================================
# SEARCH NASA EARTHDATA
# ============================================================

all_results = []


for pattern in GRANULE_PATTERNS:


    search_parameters = {
        "short_name": "SWOT_L2_HR_PIXC_D",
        "granule_name": pattern,
    }


    if TEMPORAL is not None:

        search_parameters[
            "temporal"
        ] = TEMPORAL


    results = earthaccess.search_data(
        **search_parameters
    )


    all_results.extend(
        results
    )


# ============================================================
# REMOVE DUPLICATE RESULTS
# ============================================================

# More than one search pattern can occasionally return the same
# granule. The dictionary keeps one Earthaccess object per native ID.
unique_results = {}


for result in all_results:


    native_id = result[
        "meta"
    ][
        "native-id"
    ]


    unique_results[
        native_id
    ] = result


all_results = list(
    unique_results.values()
)


# ============================================================
# DISPLAY SEARCH SUMMARY
# ============================================================

items = [
    item["meta"]["native-id"]
    for item in all_results
]


print(
    f"PIXC granules found: {len(items):,}"
)


# Display only the first granules to keep notebook output compact.
for item in items[:30]:

    print(
        item
    )


if len(items) > 30:

    print(
        f"... showing the first 30 of {len(items):,} granules."
    )


PIXC granules found: 1
SWOT_L2_HR_PIXC_050_549_179R_20260528T044523_20260528T044534_PID0_01


## 3. Download the selected PIXC granules

The final cell downloads the Earthdata search results to:

```text
~/swot_course/pixc
```

where `~` represents the current user's home directory.

This makes the notebook portable across Windows, Linux, and macOS.

The routine also:

- skips an already existing non-empty file;
- retries failed downloads;
- accepts different SWOT file extensions;
- keeps the notebook output compact;
- writes detailed download messages to a log file.


In [5]:
import contextlib
import inspect
import os
import time
from pathlib import Path

import earthaccess
from IPython.display import display


# ============================================================
# USER CONFIGURATION
# ============================================================

COURSE_DIR = Path.home() / "swot_course"


# Directory where PIXC files will be stored.
DOWNLOAD_DIR = (
    COURSE_DIR
    / "pixc"
)

DOWNLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Complete Earthaccess result objects returned by the previous cell.
FILE_RESULTS = list(
    all_results
)


# Optional limit for classroom testing.
#
# None -> download every search result.
# 10   -> download only the first 10 files.
MAX_DOWNLOADS = None
# MAX_DOWNLOADS = 10


if MAX_DOWNLOADS is not None:

    FILE_RESULTS = FILE_RESULTS[
        :MAX_DOWNLOADS
    ]


# Maximum number of attempts for each granule.
MAX_RETRIES = 3

# Waiting time before retrying a failed download.
SLEEP_SECONDS = 5


# Update the single Jupyter status line at most every few seconds.
STATUS_UPDATE_SECONDS = 3.0


# Detailed Earthaccess messages are written here.
LOG_FILE = (
    DOWNLOAD_DIR
    / "pixc_download_log.txt"
)


print(
    f"Download directory: {DOWNLOAD_DIR}"
)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_filename_from_result(
    file_result
):
    """
    Try to determine the expected local filename from an
    Earthaccess search result.
    """

    # --------------------------------------------------------
    # METHOD 1: NASA native ID
    # --------------------------------------------------------

    try:

        native_id = file_result[
            "meta"
        ][
            "native-id"
        ]


        name = Path(
            str(native_id).split("?")[0]
        ).name


        if (
            "." in name
            and len(name) > 3
        ):

            return name


    except Exception:
        pass


    # --------------------------------------------------------
    # METHOD 2: EARTHACCESS DATA LINKS
    # --------------------------------------------------------

    try:

        for link in file_result.data_links():


            name = Path(
                str(link).split("?")[0]
            ).name


            if (
                "." in name
                and len(name) > 3
            ):

                return name


    except Exception:
        pass


    return None


def write_log(
    message
):
    """
    Append one timestamped message to the download log.
    """

    timestamp = time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )


    with open(
        LOG_FILE,
        "a",
        encoding="utf-8"
    ) as log_file:

        log_file.write(
            f"[{timestamp}] {message}\n"
        )


def call_earthaccess_download(
    file_result,
    download_dir
):
    """
    Call earthaccess.download() while disabling its progress
    display whenever the installed version supports that option.
    """

    kwargs = {}


    try:

        signature = inspect.signature(
            earthaccess.download
        )


        if "show_progress" in signature.parameters:

            kwargs[
                "show_progress"
            ] = False


        if "progress" in signature.parameters:

            kwargs[
                "progress"
            ] = False


    except Exception:
        pass


    # Earthaccess usually works reliably when results are passed
    # as a one-element list.
    try:

        earthaccess.download(
            [file_result],
            str(download_dir),
            **kwargs
        )


    except TypeError:

        # Compatibility fallback for Earthaccess versions that
        # do not accept the optional progress arguments.
        earthaccess.download(
            [file_result],
            str(download_dir)
        )


def download_one(
    file_result
):
    """
    Download one PIXC granule with retry protection.

    Returns
    -------
    "ok"
        A new file was downloaded successfully.

    "skip"
        The file was already present locally.

    "fail"
        All download attempts failed.
    """

    expected_name = get_filename_from_result(
        file_result
    )


    # --------------------------------------------------------
    # SKIP EXISTING FILE
    # --------------------------------------------------------

    if expected_name is not None:


        expected_path = (
            DOWNLOAD_DIR
            / expected_name
        )


        if (
            expected_path.exists()
            and expected_path.stat().st_size > 0
        ):

            return "skip"


    # --------------------------------------------------------
    # DOWNLOAD WITH RETRIES
    # --------------------------------------------------------

    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):


        try:

            write_log(
                f"Attempt {attempt}/{MAX_RETRIES} | "
                f"{expected_name or 'unknown filename'}"
            )


            # Redirect ordinary Earthaccess stdout/stderr messages
            # to the log file instead of filling the notebook.
            with open(
                LOG_FILE,
                "a",
                encoding="utf-8"
            ) as log_file:


                with (
                    contextlib.redirect_stdout(
                        log_file
                    ),
                    contextlib.redirect_stderr(
                        log_file
                    )
                ):


                    call_earthaccess_download(
                        file_result,
                        DOWNLOAD_DIR
                    )


            # ------------------------------------------------
            # VERIFY EXPECTED FILE
            # ------------------------------------------------

            if expected_name is not None:


                expected_path = (
                    DOWNLOAD_DIR
                    / expected_name
                )


                if (
                    expected_path.exists()
                    and expected_path.stat().st_size > 0
                ):

                    write_log(
                        f"SUCCESS | {expected_path}"
                    )

                    return "ok"


            # If the expected filename could not be determined,
            # the Earthaccess call completed without raising an error.
            else:

                write_log(
                    "SUCCESS | download completed"
                )

                return "ok"


        except Exception as error:


            write_log(
                f"FAILED | attempt {attempt} | {repr(error)}"
            )


            if attempt < MAX_RETRIES:

                time.sleep(
                    SLEEP_SECONDS
                )


    return "fail"


# ============================================================
# MAIN DOWNLOAD LOOP
# ============================================================

n_total = len(
    FILE_RESULTS
)

n_ok = 0
n_skip = 0
n_fail = 0


last_status_update = 0.0


# Create ONE Jupyter output area.
status_display = display(
    f"Starting PIXC download | 0/{n_total}",
    display_id=True
)


for index, file_result in enumerate(
    FILE_RESULTS,
    start=1
):


    status = download_one(
        file_result
    )


    if status == "ok":

        n_ok += 1


    elif status == "skip":

        n_skip += 1


    else:

        n_fail += 1


    # --------------------------------------------------------
    # LIGHTWEIGHT STATUS UPDATE
    # --------------------------------------------------------

    now = time.time()


    if (
        now - last_status_update
        >= STATUS_UPDATE_SECONDS

        or status == "fail"

        or index == n_total
    ):


        current_name = (
            get_filename_from_result(
                file_result
            )
            or "unknown file"
        )


        if len(current_name) > 90:

            current_name = (
                current_name[:87]
                + "..."
            )


        status_display.update(
            (
                f"{index:,}/{n_total:,} | "
                f"OK {n_ok:,} | "
                f"SKIP {n_skip:,} | "
                f"FAIL {n_fail:,} | "
                f"{current_name}"
            )
        )


        last_status_update = now


# ============================================================
# FINAL SUMMARY
# ============================================================

files = [
    path
    for path in DOWNLOAD_DIR.glob("SWOT*.*")
    if path.is_file()
]


status_display.update(
    (
        f"DONE | "
        f"{n_total:,}/{n_total:,} | "
        f"OK {n_ok:,} | "
        f"SKIP {n_skip:,} | "
        f"FAIL {n_fail:,}"
    )
)


print()

print(
    f"Total SWOT files in folder: {len(files):,}"
)

print(
    f"Download log: {LOG_FILE}"
)


if files:

    newest = max(
        files,
        key=os.path.getmtime
    )

    print(
        f"Most recent file: {newest.name}"
    )


Download directory: /home/moreira/swot_course/pixc


'DONE | 1/1 | OK 1 | SKIP 0 | FAIL 0'

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]


Total SWOT files in folder: 1
Download log: /home/moreira/swot_course/pixc/pixc_download_log.txt
Most recent file: SWOT_L2_HR_PIXC_050_549_179R_20260528T044523_20260528T044534_PID0_01.nc
